# Tools with MCP ⏰

模型上下文协议 (MCP) 提供了一种标准化的方法，用于将 AI 代理连接到外部工具和数据源。连接到 MCP server using `langchain-mcp-adapters`.

**接入高德地图 MCP Server**

https://lbs.amap.com/api/mcp-server/summary
- 注册登录高德开放平台
- 申请Key https://lbs.amap.com/api/mcp-server/create-project-and-key

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient
import nest_asyncio
import os

# 允许在现有 asyncio 事件循环中运行新的异步代码
nest_asyncio.apply()

# 连接到高德地图 MCP Server
mcp_client = MultiServerMCPClient(
    {
         "amap-maps-streamableHTTP": {
             "transport": "streamable_http",
             "url": "https://mcp.amap.com/mcp?key="+os.getenv('AMAP_MAPS_API_KEY')
         }
    }
)

# Load tools from the MCP server
mcp_tools = await mcp_client.get_tools()
print(f"Loaded {len(mcp_tools)} MCP tools: {[t.name for t in mcp_tools]}")

Loaded 15 MCP tools: ['maps_direction_bicycling', 'maps_direction_driving', 'maps_direction_transit_integrated', 'maps_direction_walking', 'maps_distance', 'maps_geo', 'maps_regeocode', 'maps_ip_location', 'maps_schema_personal_map', 'maps_around_search', 'maps_search_detail', 'maps_text_search', 'maps_schema_navi', 'maps_schema_take_taxi', 'maps_weather']


使用高德地图MCP Server提供的工具创建一个 Agent

In [2]:
from langchain.agents import create_agent

agent_with_mcp = create_agent(
    model="openai:gpt-5",
    tools=mcp_tools,
    system_prompt="You are a helpful assistant",
)

In [3]:
result = await agent_with_mcp.ainvoke(
    {"messages": [{"role": "user", "content": "深圳的天气怎样？"}]}
)
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

深圳的天气怎样？
================================== Ai Message ==================================
Tool Calls:
  maps_weather (call_tpHQxFTQY99uUC12lEqwjPPH)
 Call ID: call_tpHQxFTQY99uUC12lEqwjPPH
  Args:
    city: 深圳
================================= Tool Message =================================
Name: maps_weather

{"city":"深圳市","forecasts":[{"date":"2025-11-23","week":"7","dayweather":"晴","nightweather":"多云","daytemp":"26","nighttemp":"18","daywind":"北","nightwind":"北","daypower":"1-3","nightpower":"1-3","daytemp_float":"26.0","nighttemp_float":"18.0"},{"date":"2025-11-24","week":"1","dayweather":"多云","nightweather":"晴","daytemp":"27","nighttemp":"16","daywind":"北","nightwind":"北","daypower":"1-3","nightpower":"1-3","daytemp_float":"27.0","nighttemp_float":"16.0"},{"date":"2025-11-25","week":"2","dayweather":"晴","nightweather":"晴","daytemp":"24","nighttemp":"15","daywind":"北","nightwind":"北","daypower":"1-3","